## 📍Paso 1: Preprocesamiento y División de los Datos

### 1.1. Carga, Limpieza y Consolidación del Target
Importamos las librerías necesarias y cargamos el dataset enriquecido. Dado que cada fila representa una víctima individual, realizamos una agregación (`groupby`) basada en el identificador único del hecho para calcular la métrica continua **'total_victimas_siniestro'**. Esta será nuestra variable objetivo (Target) para cumplir con las métricas de regresión exigidas (RMSE, MAE y R²). Además, descartamos columnas meteorológicas vacías y extraemos variables temporales útiles.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder



# 1. Cargar el dataset
df = pd.read_csv("dataset_siniestros.csv")

# Eliminar espacios invisibles en los nombres de las columnas
df.columns = df.columns.str.strip()

# 2. ¡USAR LA COLUMNA QUE YA VENÍA EN TU DATASET!
# Renombramos la columna original a 'total_victimas_siniestro'
if 'numero_total_de_victimas' in df.columns:
    df = df.rename(columns={'numero_total_de_victimas': 'total_victimas_siniestro'})

# 3. Eliminar columnas meteorológicas 100% vacías
columnas_vacias = ['snwd', 'tmin', 'tmax', 'snow', 'vsby', 'pday', 'txmn', 'txmx', 'wpgt', 'tsun']
df = df.drop(columns=[col for col in columnas_vacias if col in df.columns])

# 4. Renombrar variables climáticas para claridad
df = df.rename(columns={
    'prcp': 'lluvia_mm',
    'wspd': 'velocidad_viento',
    'temp': 'temperatura_media',
    'rhum': 'humedad_relativa'
})

# 5. Feature Engineering: Variables temporales
# (Buscamos si la fecha se llama 'fecha_siniestro' o 'fecha_siniestro_x')
col_fecha = 'fecha_siniestro_x' if 'fecha_siniestro_x' in df.columns else 'fecha_siniestro'

if col_fecha in df.columns:
    df['fecha_siniestro_limpia'] = pd.to_datetime(df[col_fecha], errors='coerce')
    df['mes'] = df['fecha_siniestro_limpia'].dt.month
    df['es_fin_de_semana'] = df['fecha_siniestro_limpia'].dt.dayofweek.isin([5, 6]).astype(int)

print("\n✅ Celda 1.1 Ejecutada: Limpieza completada.")
print(f"✅ Verificación final: ¿Existe la columna target? -> {'total_victimas_siniestro' in df.columns}")


✅ Celda 1.1 Ejecutada: Limpieza completada.
✅ Verificación final: ¿Existe la columna target? -> True


### 1.2. División de Datos y Configuración del Pipeline
Con el dataset estructurado, aislamos las características predictoras ($X$) del Target numérico ($y$). Para garantizar un flujo libre de *Data Leakage*, configuramos un `ColumnTransformer` que automatizará la imputación de nulos y el escalado/codificación de las variables de forma adaptativa. 

Finalmente, realizamos la partición reservando un **20% para el Test Set final independiente**, el cual no participará de la validación cruzada para asegurar una evaluación honesta.


In [2]:
# 1. Definición final de variables predictoras (X) y objetivo (y)
target_col = 'total_victimas_siniestro'
feature_cols = [
    'edad_victima', 'sexo_victima', 'rol_victima',       
    'anio_siniestro', 'mes', 'es_fin_de_semana',          
    'temperatura_media', 'lluvia_mm', 'velocidad_viento', 'humedad_relativa' 
]

# Aseguramos que solo usamos las columnas que existen
feature_cols = [col for col in feature_cols if col in df.columns]

X = df[feature_cols]
y = df[target_col]

# 2. Identificación de tipos de datos para el Pipeline
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()

# 3. Diseño de los transformadores
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_cols),
        ('cat', cat_transformer, cat_cols)
    ]
)

# 4. División formal (80% Entrenamiento y 20% Prueba)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("=== PASO 1 COMPLETADO ===")
print(f"Filas para entrenamiento (X_train): {X_train.shape[0]}")
print(f"Filas para evaluación final (X_test): {X_test.shape[0]}")

=== PASO 1 COMPLETADO ===
Filas para entrenamiento (X_train): 49660
Filas para evaluación final (X_test): 12416


C:\Users\Celi\AppData\Local\Temp\ipykernel_25172\2608542165.py:17: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()


### 1.3. Aislamiento del Target y Selección de Features (Evitando Data Leakage)
Para cumplir con las métricas obligatorias de la rúbrica (RMSE, MAE y R²), abordamos el problema desde una perspectiva de **regresión**, utilizando la **cantidad de víctimas** involucradas como nuestra variable objetivo continua/numérica. 

Para evitar la fuga de datos (*Data Leakage*), excluimos identificadores abstractos (como `id_siniestro`) y variables que registran eventos posteriores al accidente (como `fecha_fallecimiento`). Definimos las variables predictoras ($X$) integrando los bloques de perfil humano, contexto espacial y los factores climáticos (`temp_max`, `lluvia_mm`, etc.).

In [3]:
# Definición de la variable objetivo (Target)
target_col = 'total_victimas_siniestro'

feature_cols = [
    'edad_victima', 'sexo_victima', 'rol_victima',       # Perfil (Nota: rol_victima tiene 211 nulos, se imputarán)
    'anio_siniestro', 'mes', 'es_fin_de_semana',          # Contexto (mes y es_fin_de_semana se extraen de fecha)
    'temperatura_media', 'lluvia_mm', 'velocidad_viento', 'humedad_relativa' # Clima Real
]


# Filtramos únicamente las columnas que se encuentren presentes en el DataFrame
feature_cols = [col for col in feature_cols if col in df.columns]

X = df[feature_cols]
y = df[target_col]

print(f"Target seleccionado: '{target_col}'")
print(f"Cantidad de features asignadas a X: {len(feature_cols)}")

Target seleccionado: 'total_victimas_siniestro'
Cantidad de features asignadas a X: 9


### 1.4. Diseño del Pipeline Automatizado (ColumnTransformer)
Siguiendo las buenas prácticas recomendadas para evitar el sesgo en el modelado, implementamos un `ColumnTransformer`. Este dividirá el flujo de procesamiento según el tipo de datos:
* **Variables Numéricas:** Serán imputadas con la *mediana* para manejar valores faltantes sin resentir la distribución ante outliers, y luego serán estandarizadas mediante `StandardScaler` para que todas compartan la misma escala numérica.
* **Variables Categóricas:** Serán imputadas con la *moda* (valor más frecuente) y transformadas numéricamente mediante *One-Hot Encoding*. Configuramos `sparse_output=False` para facilitar la posterior exportación del dataset limpio a la base de datos NoSQL requerida.

In [4]:
# Separación de columnas según su tipo de dato para el pipeline
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()

# Pipeline para datos numéricos
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Pipeline para datos categóricos
cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Combinación de transformadores en el preprocesador unificado
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_cols),
        ('cat', cat_transformer, cat_cols)
    ]
)

print(f"Pipeline de preprocesamiento configurado.")
print(f"Columnas numéricas mapeadas: {num_cols}")
print(f"Columnas categóricas mapeadas: {cat_cols}")

Pipeline de preprocesamiento configurado.
Columnas numéricas mapeadas: ['es_fin_de_semana', 'temperatura_media', 'lluvia_mm', 'velocidad_viento', 'humedad_relativa']
Columnas categóricas mapeadas: ['edad_victima', 'sexo_victima', 'rol_victima']


C:\Users\Celi\AppData\Local\Temp\ipykernel_25172\1116418322.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()


### 1.5. División del Dataset (Train/Test Split)
[cite_start]Para garantizar una evaluación honesta de la capacidad de generalización del modelo, realizamos una partición de los datos disponibles en un **80% para el conjunto de entrenamiento** (Training Set) y un **20% para el conjunto de prueba** (Test Set)[cite: 33, 34]. [cite_start]El conjunto de prueba se mantendrá aislado y servirá exclusivamente para calcular las métricas finales[cite: 34]. Fijamos un `random_state` para asegurar que esta división sea completamente reproducible en futuras ejecuciones.

In [5]:
# Partición del dataset en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("--- Resumen de la División de Datos ---")
print(f"Registros totales en X_train: {X_train.shape[0]}")
print(f"Registros totales en X_test: {X_test.shape[0]}")
print(f"Proporción del Test Set: {round((X_test.shape[0] / X.shape[0]) * 100, 1)}%")
print("\n¡Paso 1 finalizado con éxito bajo estándares profesionales!")

--- Resumen de la División de Datos ---
Registros totales en X_train: 49660
Registros totales en X_test: 12416
Proporción del Test Set: 20.0%

¡Paso 1 finalizado con éxito bajo estándares profesionales!
